# Syntetisk fromhet – analysnotebook

Den här notebooken återskapar samtliga kvantitativa analyser och citatkontroller som ligger till grund för resultatavsnittet i artikeln *Syntetisk fromhet: Diskursiv konstruktion i en stor språkmodell* (Jonas Svensson, Linnéuniversitetet).

**Material:** `merged_file.csv` – 30 syntetiska narrativ genererade av Claude Sonnet 4.5 (Anthropic) den 24 oktober 2025, 10 per fromhetsnivå (LÅG, MEDEL, HÖG). Demografiska variabler slumpades fram av ett separat program (`profile_generator.py`) och infogades i prompten.

**Analysens tillkomst:** Analyserna utfördes 6–9 oktober 2026 i dialog mellan författaren och Claude (Anthropic, Cowork-miljön). Claude skrev koden och föreslog tolkningar; författaren formulerade frågorna, granskade utfallen och fattade alla tolkningsbeslut. All kvantitativ information i artikeln kommer från deterministisk kod i denna notebook – inte från språkmodellens egna uppskattningar. Se avsnittet *Metod och förbehåll* sist.

**Körning:** Kör cellerna uppifrån och ned. Cell 1 ber om att få ladda upp `merged_file.csv` (eller ange en sökväg).

## 0. Inläsning av data

In [ ]:
import os, re, math, itertools, collections
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_rows', 300)

CSV = 'merged_file.csv'   # huvudmaterialet: 30 narrativ (10 per nivå) med programgenererade profiler
if not os.path.exists(CSV):
    try:
        from google.colab import files   # endast i Colab
        print('Ladda upp merged_file.csv')
        uploaded = files.upload()
        CSV = 'merged_file.csv' if 'merged_file.csv' in uploaded else list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError('Lägg merged_file.csv i arbetskatalogen eller ändra variabeln CSV.')

d = pd.read_csv(CSV)

# Kontroll: rätt fil? Huvudmaterialet har programgenererade profilvariabler.
_kravs = {'Iteration', 'Kön', 'Bakgrund', 'Generation', 'Utbildning', 'Svar', 'Fromhetsnivå'}
if not _kravs.issubset(d.columns) or len(d) != 30:
    raise ValueError(
        f"{CSV} ser inte ut att vara huvudmaterialet (merged_file.csv): {len(d)} rader, kolumner {list(d.columns)}. "
        "Ladda upp merged_file.csv. Den tidiga körningen där Claude själv valde demografin har andra kolumner "
        "och analyseras inte i denna notebook.")
ORDER = ['LÅG', 'MEDEL', 'HÖG']
d['Fromhetsnivå'] = pd.Categorical(d['Fromhetsnivå'], ORDER, ordered=True)
d = d.sort_values(['Fromhetsnivå', 'Iteration']).reset_index(drop=True)
d['id'] = d['Fromhetsnivå'].astype(str) + ' ' + d['Iteration'].astype(str)
d['t'] = d['Svar'].str.lower()
d['tok'] = d['t'].apply(lambda s: re.findall(r"[a-zåäöéüç']+", s))
d['ord'] = d['tok'].str.len()
print(d.shape)
d.groupby('Fromhetsnivå', observed=True)['ord'].agg(['count', 'sum', 'mean', 'std', 'min', 'max']).round(1)

## 1. Demografi per fromhetsnivå (de slumpade profilvariablerna)

In [ ]:
for col in ['Kön', 'Bakgrund', 'Generation', 'Civilstånd', 'Bostadsort']:
    print(f'\n--- {col}')
    print(pd.crosstab(d[col], d['Fromhetsnivå']))
print('\nÅlder:'); print(d.groupby('Fromhetsnivå', observed=True)['Ålder'].describe().round(1))

## 2. Hjälpfunktioner

* `kwic()` – visar varje träff i sitt sammanhang (*keyword in context*). Används för att manuellt granska att en regex fångar rätt betydelse.
* `count_table()` – antal träffar och antal narrativ (av 10) per grupp.
* `per_doc()` – vilka narrativ som innehåller mönstret.

In [ ]:
def kwic(pattern, groups=ORDER, width=90, flags=re.I):
    for _, r in d[d['Fromhetsnivå'].isin(groups)].iterrows():
        for m in re.finditer(pattern, r['Svar'], flags):
            s = r['Svar'][max(0, m.start()-width): m.end()+width].replace('\n', ' ')
            print(f"[{r['id']} {r['Kön'][0]}{r['Ålder']}] …{s}…")

def count_table(patterns, flags=re.I):
    rows = []
    for name, p in patterns.items():
        row = {'mönster': name}
        for g in ORDER:
            c = d.loc[d['Fromhetsnivå'] == g, 'Svar'].str.count(re.compile(p, flags))
            row[f'{g} antal'] = int(c.sum())
            row[f'{g} narrativ'] = int((c > 0).sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index('mönster')

def per_doc(pattern, flags=re.I):
    c = d['Svar'].str.count(re.compile(pattern, flags))
    return d.loc[c > 0, 'id'].tolist()

## 3. Termlexikon och frekvenstabell

Reguljära uttryck för religiös terminologi, praktiker och teman. Lexikonet är skrivet för att fånga svenska böjningsformer och vanliga translitterationsvarianter. **Varje mönster har granskats med `kwic()`**; kända felträffar och avgränsningar är kommenterade.

In [ ]:
TERMS = {
    # ord för fromhet
    'from (alla former)':          r'\bfrom(m|ma|maste|het\w*|t)?\b',
    # religiösa utrop
    'alhamdulillah':               r'alhamdulill?ah?',
    'mashallah':                   r'mash.?allah|masha.?allah',
    'subhanallah':                 r'subhan.?allah',
    'bismillah':                   r'bismillah',
    # bön
    'fajr / morgonbön':            r'\bfajr|morgonbön',
    'zuhr/dhuhr / middagsbön':     r'\bzuhr|\bdhuhr|\bduhr|\bdhur|middagsbön',
    'asr':                         r'\basr\b|eftermiddagsbön',
    'maghrib':                     r'maghrib',
    'isha':                        r"\bisha\b|ishabön|kvällsbön",
    'dhikr':                       r'\bdhikr',
    "du'a":                        r"\bdu.?a\b|\bdua\b",
    'witr':                        r'\bwitr',
    'tarawih':                     r'taraw[ie]h',
    'slå ihop böner':              r'slår (jag )?ihop|kombinerar',
    # högtider och fasta
    'ramadan':                     r'ramadan',
    'eid / id':                    r'\beid\b|\bid\b|eid-',
    'iftar':                       r'iftar',
    'frivillig fasta (veckodagar)': r'fast\w*[^.]{0,40}(måndag|tisdag)\w* och (torsdag|onsdag)|(måndag|tisdag)\w* och (torsdag|onsdag)\w*[^.]{0,40}fast',
    'jul':                         r'\bjul\b',
    # mat och dryck
    'halal':                       r'halal',
    'fläsk/gris':                  r'fläsk|\bgris\w*|bacon',
    'alkohol (explicit)':          r'alkohol|\bvinet\b|tackade nej till öl|inte dricker på',
    'alkoholfri öl/cider':         r'alkoholfri',
    'te/çay':                      r'\bte\b|çay',
    'kaffe/fika':                  r'kaffe|\bfika\b',
    # kläder
    'slöja/hijab':                 r'hijab\w*|slöj\w*',
    'jilbab/niqab':                r'jilbab|niqab',
    'skägg':                       r'skägg',
    # källor och lärande
    'Koranen':                     r'koran\w*|qur.?an',
    'hadith':                      r'had[ie]th',
    'tafsir':                      r'tafs[ie]r',
    'sunna':                       r'\bsunn?a',
    'sira':                        r'\bsira\b',
    'fiqh/jurisprudens':           r'\bfiqh|jurisprudens',
    'juz':                         r'\bjuz',
    'ayat al-kursi':               r'ayat al-kursi',
    'tajwid':                      r'tajw\w*',
    'halaqa/studiecirkel':         r'halaq\w*|studiecirk\w*|kunskapscirk\w*|kvinnocirk\w*',
    'shuyukh':                     r'shuyukh',
    'imam':                        r'imam\w*',
    # platser och socialt
    'moské':                       r'mosk\w*',
    'restaurang':                  r'restaurang',
    'bröder/systrar (alla bet.)':  r'\bbröder\w*|\bbroder\b|\bsystrar\w*|\bsyster\b',
    'vänner/kompisar':             r'\bvänner\b|\bkompisar\b|\bväninnor\b',
    # sinnesstämning och berättarformler
    'ärlighets-/bekännelseformel': r'(ska|måste) (jag )?(vara )?ärlig|jag är ärlig|om jag ska vara (helt )?ärlig|ärligt talat|skulle ljuga|måste (jag )?erkänna|får jag vara ärlig',
    'vana/kultur snarare än relig.': r'(mer|mest) (av )?(en )?(vana|kulturell)|vana än|kultur(ell)? (grej|högtid)|än (något )?(djupt )?religiöst',
    'sitter i ryggmärgen':         r'ryggmärg',
    'stress/trött':                r'stress|trött|orka',
    'prövning':                    r'prövning|testar',
    'huset tyst / staden sover':   r'(huset|hela huset) (är )?(tyst|sover)|sover fortfarande|fortfarande sover|ännu sover',
    'mellan två världar':          r'mellan två (världar|kulturer)|mitt emellan',
    'god medborgare / samhället':  r'medborgare|samhället',
    'försöker':                    r'försök',
    'tacksam':                     r'tacksam',
    'frid':                        r'\bfrid\b',
    # frånvarokontroller
    'sufi/tariqa/mawlid':          r'sufi|tariq|mawlid|mevlid|ziyara',
    'salafi-markörer':             r'salaf|bid.?a\b|tawh?id|aqid|shirk|manhaj',
    'folkreligion (amulett, jinn…)': r'amulett|taw[ie]z|nazar|onda ögat|jinn|ruqya',
}
termtab = count_table(TERMS)
termtab

### 3.1 Kontroll av enskilda påståenden

Varje påstående i resultatavsnittet har kontrollerats genom att lista *vilka* narrativ som innehåller mönstret och läsa träffarna i sitt sammanhang. Exempel (byt mönster och grupp efter behov):

In [ ]:
for name in ['fajr / morgonbön', 'halal', 'fläsk/gris', 'eid / id', 'restaurang', 'hadith', 'halaqa/studiecirkel']:
    print(f'{name:25s}', {g: [i for i in per_doc(TERMS[name]) if i.startswith(g)] for g in ORDER})

In [ ]:
kwic(r'fläsk|\bgris\w*|bacon', groups=['LÅG'])

### 3.2 Kaféer och fika – manuell avgränsning

Regexen `\bfika\b` utesluter *specifika*, *fikarummet* och *fikaraster*. Kvar granskas manuellt; träffar som inte avser kafé/fika som social mötesplats (t.ex. en jobbansökan till ett café, skolans cafeteria) exkluderas uttryckligen nedan.

In [ ]:
CAFE = r'\bcaf[eé](er|erna)?\b|\bkafé(er|erna)?\b|\bfika\b'
kwic(CAFE)
EXCLUDE_CAFE = {'MEDEL 6'}   # "ett personligt brev på svenska till ett café där jag sökte jobb"
cafe_docs = [i for i in per_doc(CAFE) if i not in EXCLUDE_CAFE]
print('\nKafé/fika som social plats:', {g: [i for i in cafe_docs if i.startswith(g)] for g in ORDER})

### 3.3 Slöjan bland de kvinnliga informanterna

In [ ]:
SLOJA = r'hijab\w*|slöj\w*|täcka håret|jilbab|niqab'
w = d[d['Kön'] == 'Kvinna'].copy()
w['nämner slöja'] = w['Svar'].str.contains(SLOJA, case=False, regex=True)
print(pd.crosstab(w['Fromhetsnivå'], w['nämner slöja'], margins=True))
kwic(SLOJA)

## 4. Allah och Gud

Fristående *Allah* skiljs från Allah i formler (alhamdulillah m.fl.). *Gud* räknas som gud/guds/guden (inte *gudsfruktan*). Testerna görs på frekvens per 1 000 ord per narrativ (Kruskal–Wallis, Mann–Whitney med Bonferroni) och på förekomst ja/nej (Fisher).

In [ ]:
P = {'Allah (fristående)': r"\ballah(s)?\b(?!u)",
     'Allah i formler':    r"allahu akbar|subhan.?allah|alhamdulill?ah|mash.?allah|in.?sha.?allah|bismillah",
     'Gud':                r"\bgud(s|en)?\b"}
for k, p in P.items():
    d[k] = d['t'].str.count(p)
d['Allah totalt'] = d['Allah (fristående)'] + d['Allah i formler']

summary = []
for c in ['Allah (fristående)', 'Allah i formler', 'Allah totalt', 'Gud']:
    for g in ORDER:
        s = d[d['Fromhetsnivå'] == g]
        summary.append([c, g, int(s[c].sum()), int((s[c] > 0).sum()),
                        round(s[c].sum() / s['ord'].sum() * 1000, 2), round(s[c].mean(), 2), round(s[c].std(), 2)])
pd.DataFrame(summary, columns=['mått', 'grupp', 'antal', 'narrativ>0', 'per 1000 ord', 'medel', 'SD'])

In [ ]:
def group_tests(col):
    r = [d.loc[d['Fromhetsnivå'] == g, col] / d.loc[d['Fromhetsnivå'] == g, 'ord'] * 1000 for g in ORDER]
    H, p = stats.kruskal(*r)
    print(f'{col}: Kruskal–Wallis H={H:.2f}, p={p:.2g}')
    for a, b in itertools.combinations(range(3), 2):
        U, pu = stats.mannwhitneyu(r[a], r[b], alternative='two-sided')
        print(f'   {ORDER[a]} vs {ORDER[b]}: U={U:.0f}, p={pu:.2g}, Bonferroni p={min(1, pu*3):.2g}')
    tab = np.array([[(d.loc[d['Fromhetsnivå'] == g, col] > 0).sum(), (d.loc[d['Fromhetsnivå'] == g, col] == 0).sum()] for g in ORDER])
    for a, b in itertools.combinations(range(3), 2):
        print(f'   Fisher förekomst {ORDER[a]} vs {ORDER[b]}: p={stats.fisher_exact(tab[[a, b]])[1]:.2g}')

for c in ['Allah totalt', 'Allah (fristående)', 'Gud']:
    group_tests(c)

hog = [d.loc[d['Fromhetsnivå'] == 'HÖG', 'Allah (fristående)'].sum(), d.loc[d['Fromhetsnivå'] == 'HÖG', 'Gud'].sum()]
rest = [d.loc[d['Fromhetsnivå'] != 'HÖG', 'Allah (fristående)'].sum(), d.loc[d['Fromhetsnivå'] != 'HÖG', 'Gud'].sum()]
print('\nAndel Allah av Allah+Gud – HÖG', hog, 'vs LÅG+MEDEL', rest, ': Fisher p =', round(stats.fisher_exact([hog, rest])[1], 4))
kwic(r'\bgud(s|en)?\b|\ballah\b', groups=['LÅG', 'MEDEL'])

## 5. Ordval: *inte/aldrig/sällan* respektive *alltid/varje*

In [ ]:
def wordcounts(words):
    out = {}
    for g in ORDER:
        toks = [t for lst in d.loc[d['Fromhetsnivå'] == g, 'tok'] for t in lst]
        c = collections.Counter(toks)
        out[g] = {w: c[w] for w in words} | {'totalt antal ord': len(toks)}
    return pd.DataFrame(out)
wordcounts(['inte', 'aldrig', 'sällan', 'alltid', 'varje'])

### 5.1 Claudes ursprungliga sentimentlistor – nedbrutna

I den tidiga analysen föreslog Claude att högfromhetsnarrativen var mer positivt och lågfromhetsnarrativen mer negativt laddade. Här räknas Claudes egna ordlistor, och listorna delas sedan upp i frekvensord respektive känsloord.

In [ ]:
LISTS = {
  'Claudes negativa lista': r'\b(inte|aldrig|sällan|missar|svårt|svår|problem|dåligt samvete|besviken|besvikna|skuld|dålig)\b',
  'Claudes positiva lista': r'\b(alltid|varje|älskar|älska|glädje|glad|tacksam|stolt|alhamdulillah|mashallah|subhanallah)\b',
  'frekvensord negativa':   r'\b(inte|aldrig|sällan)\b',
  'frekvensord positiva':   r'\b(alltid|varje)\b',
  'känsloord negativa':     r'dåligt samvete|skuld|skäm|besvik|ledsen|ångest|oro|rädsla|ensam|tung|jobbig|dömd|vemod|stress|trött',
  'känsloord positiva':     r'glädje|\bglad|älsk|tacksam|stolt|frid|\bro\b|lugn|mysig|skön|värde|berik|välsign|värmer',
}
rows = []
for k, p in LISTS.items():
    rows.append([k] + [round(d.loc[d['Fromhetsnivå'] == g, 't'].str.count(p).sum() / d.loc[d['Fromhetsnivå'] == g, 'ord'].sum() * 1000, 1) for g in ORDER])
pd.DataFrame(rows, columns=['lista (per 1000 ord)'] + ORDER)

## 6. Icke-religiöst ordförråd

Alla ord som förekommer i minst fem narrativ, utom religiös terminologi (se `REL`), testas med Kruskal–Wallis på frekvens per 1 000 ord. **Obs!** Cirka 400 ord testas; enstaka träffar runt p ≈ 0,01 kan vara slumpmässiga. Kategorierna i 6.2 testas som helheter och är därför mer robusta.

In [ ]:
REL = re.compile(r"^(allah\w*|gud\w*|bön\w*|be|ber|bad|bett|moské\w*|mosk\w*|koran\w*|fajr|zuhr|dhuhr|duhr|dhur|asr|maghrib|isha|witr|dhikr|du'?a|tafsir|hadith\w*|sunn?a\w*|halal\w*|haram|ramadan|eid|id|fast\w*|iftar|tarawih|jum\w*|khutba\w*|imam\w*|islam\w*|muslim\w*|religi\w*|tro|tron|troende|tror|from\w*|halaq\w*|studiecirkel\w*|systrar\w*|syster\w*|bröder\w*|broder|profet\w*|hijab\w*|slöja\w*|alhamdulillah|subhanallah|mashallah|bismillah|inshallah|wudu\w*|tvagning\w*|iman|ibadah?|umma\w*|tajw\w*|sura\w*|ayat|vers\w*|sadaq\w*|baraka|fläsk|gris\w*|alkohol\w*|fredagsbön\w*|bönerna|bönen|böner|bönerum\w*|fredagspredikan|koranskola|dars|dawah|shuyukh|sahabah|sira|fiqh|juz|jilbab|nasheeds|akhirah|dunya|din|taqwa|fardh|wajib|shariah|jihad|kvinnocirkel\w*|kunskapscirkeln|helig\w*|andlig\w*|välsign\w*|prövning\w*|vägledning|ishabönen|dhurbönen)$")

docfreq = {g: collections.Counter() for g in ORDER}
cnt = {g: collections.Counter() for g in ORDER}
for _, r in d.iterrows():
    cnt[r['Fromhetsnivå']].update(r['tok']); docfreq[r['Fromhetsnivå']].update(set(r['tok']))
vocab = [w for w in set().union(*cnt.values()) if not REL.match(w) and sum(docfreq[g][w] for g in ORDER) >= 5]
res = []
for w in vocab:
    rates = [d.loc[d['Fromhetsnivå'] == g].apply(lambda r: r['tok'].count(w) / r['ord'] * 1000, axis=1) for g in ORDER]
    try:
        H, p = stats.kruskal(*rates)
    except ValueError:
        continue
    res.append([w, p] + [docfreq[g][w] for g in ORDER] + [cnt[g][w] for g in ORDER])
keyness = pd.DataFrame(res, columns=['ord', 'p'] + [f'{g} narrativ' for g in ORDER] + [f'{g} antal' for g in ORDER]).sort_values('p')
print(len(keyness), 'ord testade')
keyness.head(60)

### 6.2 Ordkategorier (testade som helheter)

In [ ]:
CATS = {
 'jag/mig/min':                 ['jag', 'mig', 'min', 'mitt', 'mina'],
 'vi/oss/vår':                  ['vi', 'oss', 'vår', 'vårt', 'våra'],
 'garderingar':                 ['kanske', 'nog', 'lite', 'ganska', 'typ', 'liksom', 'egentligen', 'väl'],
 'men':                         ['men'],
 'känns/känner':                ['känns', 'kändes', 'känner', 'kände'],
 'veckodagar':                  ['måndagar', 'tisdagar', 'onsdagar', 'torsdagar', 'fredagar', 'lördagar', 'söndagar',
                                 'måndag', 'tisdag', 'onsdag', 'torsdag', 'fredag', 'lördag', 'söndag'],
 'tid på dygnet':               ['morgon', 'morgonen', 'morgnarna', 'kväll', 'kvällen', 'kvällarna', 'natt', 'natten', 'gryningen', 'soluppgången'],
 'vänner/kompisar':             ['vänner', 'vän', 'kompisar', 'kompis', 'väninnor'],
 'lära/förstå/studera':         ['lära', 'lär', 'lärt', 'förstå', 'förståelse', 'studera', 'studerar', 'fördjupa', 'kunskap'],
 'identitetsformler':           ['styr', 'bakgrund', 'vem'],
}
LV = {'LÅG': 0, 'MEDEL': 1, 'HÖG': 2}
rows = []
for k, ws in CATS.items():
    r = d.apply(lambda x: sum(x['tok'].count(w) for w in ws) / x['ord'] * 1000, axis=1)
    m = [r[d['Fromhetsnivå'] == g].mean() for g in ORDER]
    H, p = stats.kruskal(*[r[d['Fromhetsnivå'] == g] for g in ORDER])
    rho, pr = stats.spearmanr(d['Fromhetsnivå'].astype(str).map(LV), r)
    rows.append([k] + [round(x, 1) for x in m] + [round(p, 5), round(rho, 2)])
pd.DataFrame(rows, columns=['kategori (per 1000 ord)'] + ORDER + ['Kruskal–Wallis p', 'Spearman ρ'])

## 7. Narrativens inledningar

In [ ]:
for g in ORDER:
    print('---', g)
    for _, r in d[d['Fromhetsnivå'] == g].iterrows():
        s = re.sub(r'^#.*\n', '', r['Svar'].strip()).strip()
        print('  ', r['id'], '|', s.split('.')[0][:120])

## 8. Läs ett helt narrativ

In [ ]:
def show(level, iteration):
    r = d[(d['Fromhetsnivå'] == level) & (d['Iteration'] == iteration)].iloc[0]
    print(r[['Ålder', 'Kön', 'Bakgrund', 'Generation', 'Civilstånd', 'Barn', 'Sysselsättning', 'Bostadsort']].to_string())
    print(); print(r['Svar'])
show('HÖG', 1)

## 9. Explorativa testkörningar K1–K6

Sex kontrollkörningar (n = 10 vardera) där endast instruktionsraden om informantens religiositet ändrades. Profilgeneratorn, modellen (Claude Sonnet 4.5) och promptens övriga text var desamma som i huvudstudien. Körningarna är explorativa och utformades successivt; de prövar inte i förväg formulerade hypoteser.

| Körning | Instruktionsrad (förkortad) |
|---|---|
| K1 | "Din nivå av folklig fromhet är: HÖG" |
| K2 | "Din nivå av folkreligös fromhet är: HÖG" (stavningen *folkreligös* som i den använda promptmallen) |
| K3 | "Din religiositet tar sig främst folkreligiösa uttryck." (ingen fromhet, ingen nivå) |
| K4 | K3 + "och religionen är mycket viktig i ditt liv och präglar din vardag starkt" |
| K5 | K4 + exempel: kärlek till och vördnad för profeten Muhammed, helgongravar och heliga platser, sufiska ritualer, personligt band till en andlig vägledare |
| K6 | "Din nivå av fromhet är: HÖG." + samma exempel som K5 |

Filerna har en enhetlig struktur: `Körning`, `Iteration`, profilvariablerna, `Fromhetsnivå_i_prompt` (HÖG där nivån ingår i prompten, annars tom), `Instruktion` (den varierade raden), `Prompt` (hela prompten), `Prompt_källa`, `Modell` och `Svar`. För K3–K6 är prompten **loggad** vid genereringen. K1 och K2 genererades innan promptloggningen infördes; deras prompter är **rekonstruerade** från promptmallen och respektive profilvärden. Metoden validerades genom att samma rekonstruktion återskapade samtliga 50 loggade prompter ordagrant. En parallell körning med K4-prompten (`K41.csv`) gav samma mönster och ingår inte här.

In [ ]:
# Läs in K1–K6. Filerna söks i arbetskatalogen eller i "Runs Jämförelsetester/"; annars laddas de upp (Colab).
KRUNS = ['K1', 'K2', 'K3', 'K4', 'K5', 'K6']
def _find(name):
    for cand in [name, os.path.join('Runs Jämförelsetester', name), os.path.join('..', 'Runs Jämförelsetester', name)]:
        if os.path.exists(cand):
            return cand
    return None

missing = [k for k in KRUNS if _find(k + '.csv') is None]
if missing:
    try:
        from google.colab import files
        print('Ladda upp:', ', '.join(m + '.csv' for m in missing))
        files.upload()
    except ImportError:
        raise FileNotFoundError('Saknar ' + ', '.join(missing) + '. Lägg filerna i arbetskatalogen.')

kd = pd.concat([pd.read_csv(_find(k + '.csv')).assign(Körning=k) for k in KRUNS], ignore_index=True)
kd['id'] = kd['Körning'] + ':' + kd['Iteration'].astype(str)

# Jämförelsegrupper ur huvudstudien
hd = d[['Iteration', 'Svar']].copy()
hd['Körning'] = 'Huvud-' + d['Fromhetsnivå'].astype(str)
hd['id'] = hd['Körning'] + ':' + hd['Iteration'].astype(str)

alla = pd.concat([hd, kd[['Iteration', 'Svar', 'Körning', 'id']]], ignore_index=True)
GRUPPER = ['Huvud-LÅG', 'Huvud-MEDEL', 'Huvud-HÖG'] + KRUNS
alla['Körning'] = pd.Categorical(alla['Körning'], GRUPPER, ordered=True)
alla['ord_split'] = alla['Svar'].str.split().str.len()   # enkel ordräkning (blanksteg) för detta avsnitt
print(alla.groupby('Körning', observed=True)['ord_split'].agg(['count', 'mean']).round(0))

def kwic_k(pattern, runs=KRUNS, width=90, flags=re.I):
    """Som kwic(), men för testkörningarna."""
    for _, r in kd[kd['Körning'].isin(runs)].iterrows():
        for m in re.finditer(pattern, r['Svar'], flags):
            s = r['Svar']
            print(f"{r['id']:6s} …{s[max(0, m.start()-width):m.start()]}[[{m.group(0)}]]{s[m.end():m.end()+width]}…".replace('\n', ' '))

### 9.1 Islamisk-arabisk terminologi

Fråga: används en högre nivå av islamisk-arabisk terminologi när *fromhet* på hög nivå ingår i prompten (Huvud-HÖG, K1, K2, K6) än när ordet saknas (K3, K4, K5)?

Lexikonet `ARAB` delar in termerna i fyra kategorier: **formler** (inshallah, alhamdulillah, ﷺ …), **normativ** terminologi (bönenamn, wudu, sunna, hadith, tafsir, halaqa, taqwa …), **sufisk/devotionell** (dhikr, shaykh, tariqa, wird, qasida, mawlid, salawat, ziyara, baraka …) och **folkreligiös** (taweez, nazar, hamsa, ruqya, bakhour). Svenska ord som *bön*, *fasta* och *moské* räknas inte.

Huvudmåttet är **antal olika termer per text** (påverkas inte av upprepningar som "Sheikh Ahmed" tio gånger). Som komplement anges träffar per 1 000 ord. Kända falska träffar har uteslutits: *din* ("din tro") räknas inte som *din*/*deen*, *Aisha* inte som *isha*, och *rak'a* kräver formen *rak'at*.

In [ ]:
ARAB = {
 'formler': [r'insh?a.?allah', r'masha.?allah', r'alhamdulillah?', r'bismillah', r'subhan.?allah', r'allahu?.?akbar',
             r'astaghfirullah', r'jazak', r'barak.?allahu?', r'salam aleikum|assalamu', r'ﷺ', r'sallallahu', r'subhanahu', r'wallah'],
 'normativ': [r'fajr', r'dhuhr|zuhr|duhr', r'\basr\b', r'maghrib', r'\bisha\b', r'witr', r'tahajjud', r'wud[hu]u?\b|wudu', r'sujud',
              r'\brak.?a.?at', r'jum.?u?.?ah?\b|jumm?a\b|djumah', r'khutba', r'sunn?ah?\b|sunna-', r'halal', r'haram', r'halaq|halqa',
              r'tafsir', r'hadith', r'fiqh', r'tajw[ei]e?d', r'hafiz', r'juz\b', r'surah?\b|sura-', r'\bayah?\b|ayat\b', r'ibadah?',
              r'taqwa', r'\biman\b', r'\bdeen\b', r'\bummah?\b', r'da.?wah?\b', r'sadaq', r'zakat', r'iftar', r'suhoor|suhur', r'tarawih',
              r'dua\b|du.?a.?a?\b|duas', r'qibla', r'mahram', r'hijab', r'niqab', r'abaya', r'jilbab', r'akhirah', r'jannah', r'tawhid',
              r'shir[kq]', r'bid.?a', r'istighfar', r'tasbih|tasbeeh', r'adhan|azan'],
 'sufi_devotion': [r'dhikr|zikr', r'tariq', r'sh?ay?kh|sheikh', r'murshid', r'wird|awrad', r'wazifa', r'sohbet|suhba', r'qasid|qasaid',
                   r'mawlid|maulid', r'salawat|durood', r'dala.?il', r'burda', r'nasheed', r'ziyar', r'awliya|wali\b|waliallah',
                   r'baraka?t?|barka', r'nafs\b', r'tasawwuf', r'ihsan', r'karamat', r'adab\b', r'ghaflah|kibr', r'maqam', r'tekke',
                   r'ilahi', r'halvat'],
 'folk': [r'ta.?wiz|taweez', r'nazar', r'hamsa|khamsa', r'muska', r'ruq|roqya', r'bakh|buhur', r'jinn'],
}

def typer(text, cats=ARAB):
    return sum(1 for c in cats for p in cats[c] if re.search(p, text, re.I))

def traffar(text, cat):
    return sum(len(re.findall(p, text, re.I)) for p in ARAB[cat])

alla['olika termer'] = alla['Svar'].apply(typer)
alla['olika normativa'] = alla['Svar'].apply(lambda t: typer(t, {'n': ARAB['normativ']}))
for c in ARAB:
    alla['träffar ' + c] = alla['Svar'].apply(lambda t, c=c: traffar(t, c))

g = alla.groupby('Körning', observed=True)
tab = pd.DataFrame({
    'olika termer/text': g['olika termer'].mean(),
    'min': g['olika termer'].min(),
    'max': g['olika termer'].max(),
    'olika normativa/text': g['olika normativa'].mean(),
})
ordsum = g['ord_split'].sum()
for c in ARAB:
    tab[c + ' /1000 ord'] = g['träffar ' + c].sum() / ordsum * 1000
tab['totalt /1000 ord'] = sum(g['träffar ' + c].sum() for c in ARAB) / ordsum * 1000
tab.round(1)

In [ ]:
# Test: K1/K2/K6 (fromhet: HÖG) mot K3/K4/K5 (ordet saknas)
med = alla[alla['Körning'].isin(['K1', 'K2', 'K6'])]
utan = alla[alla['Körning'].isin(['K3', 'K4', 'K5'])]

def perm_test(a, b, n=20000, seed=1):
    a, b = np.asarray(a, float), np.asarray(b, float)
    obs = a.mean() - b.mean()
    pool = np.concatenate([a, b]); rng = np.random.default_rng(seed); k = 0
    for _ in range(n):
        rng.shuffle(pool)
        if abs(pool[:len(a)].mean() - pool[len(a):].mean()) >= abs(obs):
            k += 1
    return obs, (k + 1) / (n + 1)

for col in ['olika termer', 'olika normativa']:
    diff, p_perm = perm_test(med[col], utan[col])
    U, p_mw = stats.mannwhitneyu(med[col], utan[col], alternative='two-sided')
    print(f"{col:16s} med fromhet {med[col].mean():5.1f}  utan {utan[col].mean():5.1f}  "
          f"permutationstest p {'< 0.0001' if p_perm < 1e-4 else '= %.4f' % p_perm}  Mann–Whitney U = {U:.0f}, p = {p_mw:.4f}")

# Parvis: K5 mot K6 (samma exempel, skiljer sig bara i 'fromhet: HÖG')
for col in ['olika termer', 'olika normativa']:
    a = alla.loc[alla['Körning'] == 'K6', col]; b = alla.loc[alla['Körning'] == 'K5', col]
    U, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    print(f"K6 mot K5, {col:16s}: {a.mean():.1f} mot {b.mean():.1f}, Mann–Whitney p = {p:.3f}")

In [ ]:
# Vanligaste termerna per grupp (antal träffar) – visar vilket ordförråd som bär tätheten
for k in ['Huvud-HÖG', 'K1', 'K2', 'K3', 'K4', 'K5', 'K6']:
    cnt = collections.Counter()
    for t in alla.loc[alla['Körning'] == k, 'Svar']:
        for cat, pats in ARAB.items():
            for p in pats:
                for m in re.findall(p, t, re.I):
                    cnt[m.lower() if isinstance(m, str) and m else p] += 1
    print(f"{k:10s}", ', '.join(f"{w} ({n})" for w, n in cnt.most_common(10)))

**Tolkning.** Med *fromhet: HÖG* i prompten (K1, K2, K6) används i genomsnitt 12–13 olika islamisk-arabiska termer per narrativ, i nivå med huvudstudiens högfromhetsgrupp; utan begreppet (K3–K5) 3–10. Skillnaden ligger främst i den **normativa** terminologin. I K5 och K6 är tätheten per 1 000 ord lika hög som i Huvud-HÖG, men den bärs av ett **sufiskt** ordförråd (shaykh, dhikr, qasida, ﷺ). Från K5 till K6 ökar den normativa terminologin (fajr, dhuhr, sunna, jumu'a), men når inte K1/K2-nivå.

Förbehåll: lexikonet är konstruerat för denna studie och inte uttömmande; stavningsvarianter kan missas; granska tveksamma termer med `kwic_k()`, t.ex. `kwic_k(r'\bayah?\b|ayat\b')`.

## Metod och förbehåll

1. **Generering (oktober 2025).** Narrativen genererades av Claude Sonnet 4.5 via API med prompten som återges i artikeln. Demografiska variabler slumpades fram med viss viktning och infogades i prompten.
2. **Tidig explorativ analys (oktober 2025).** Claude läste samtliga narrativ och föreslog mönster. Flera av dessa förslag visade sig vid kontroll vara felaktiga eller överdrivna, bland annat en hallucinerad uppgift om att en samboende informant kallade sin partner "make".
3. **Systematisk kontroll (oktober 2026).** Varje påstående i resultatavsnittet kontrollerades mot data med denna kod. Claude (Cowork, Anthropic) skrev koden och föreslog tolkningar; författaren granskade samtliga utfall och citat och fattade alla tolkningsbeslut. Alla siffror i artikeln är framräknade av koden, inte av språkmodellen.
4. **Lexikon.** Termlistorna är reguljära uttryck som täcker svenska böjningsformer och translitterationsvarianter. De har granskats med `kwic()`. Kända avgränsningar: *specifika* m.fl. utesluts genom ordgränser; kafé som jobbsökningsmål (MEDEL 6) utesluts manuellt; *gudsfruktan* räknas inte som *Gud*.
5. **Statistik.** n = 10 per grupp. Icke-parametriska test (Kruskal–Wallis, Mann–Whitney, Fisher, Spearman) används. I avsnitt 6 testas många ord; enskilda ord med p kring 0,01 ska läsas som indikationer, inte fynd.
6. **Cirkularitet.** Samma modellfamilj genererade och (initialt) analyserade materialet. Detta motverkas genom att alla kvantitativa påståenden här vilar på deterministisk räkning och att alla citat hämtas direkt ur CSV-filen.

7. **Testkörningar K1–K6 (oktober 2026).** Sex explorativa varianter av prompten, n = 10 vardera, utformade successivt efter föregående resultat (avsnitt 9). Prompt och modell sparades för K3–K6; för K1–K2, som genererades före loggningen, är prompten rekonstruerad (se avsnitt 9). Resultaten redovisas som indikationer, inte som hypotesprövning.

Miljö som analysen ursprungligen kördes i: Python 3, pandas 3.0, SciPy 1.18, NumPy 2.5. Notebooken är skriven för att även fungera med Colabs standardversioner.